In [ ]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor,Lambda
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch import nn

In [ ]:
train_data = datasets.FashionMNIST(
    root = r"<dataset_path>",
    train=True,
    download=False,
    transform=ToTensor(),
)

In [ ]:
test_data = datasets.FashionMNIST(
    root = r"C:\Users\Rishee Keshavan\Documents\ML\FashionMNIST\data",
    train=False,
    download=False,
    transform=ToTensor(),
)

In [ ]:
images = train_data.data
labels = train_data.targets

In [ ]:
figure = plt.figure(figsize=(9,9))
for i in range(1,10):
    rand_idx = torch.randint(0,len(labels),size=(1,1)).item()
    img,lbl = train_data[rand_idx]
    figure.add_subplot(3,3,i)
    plt.axis("off")
    plt.imshow(img.squeeze(),cmap="gray")
plt.show()

In [ ]:
train_dataloader = DataLoader(train_data,batch_size=64,shuffle=True)
test_dataloader = DataLoader(test_data,batch_size=64,shuffle=True)

In [ ]:
class MyNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
                    nn.Conv2d(1,1,(5,5)),
                    nn.MaxPool2d((2,2),stride=1),
                    nn.Flatten(),
                    nn.Linear(23*23,144),
                    nn.ReLU(),
                    nn.Linear(144,144),
                    nn.ReLU(),
                    nn.Linear(144,10)
        )
    def forward(self,X):
        logits = self.linear_relu_stack(X)
        return logits

In [ ]:
model = MyNN()

In [ ]:
epochs = 5
learning_rate = 1e-2
batch_size = 32

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

In [ ]:
def train_loop(dataloader,model,loss_fn,optimizer):
    size = len(dataloader.dataset)
    for batch, (X,y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred,y)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (batch%100)==0:
            loss = loss.item()
            curr = batch*batch_size+len(X)
            print(f"loss: {loss:>7f} at iter [{curr:>5d}/{size:>5d}]")

In [ ]:
def test_loop(dataloader,model,loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X,y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1)==y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error:\nAccuracy: {100*correct:>4f}%, Avg loss: {test_loss:>8f}")

In [ ]:
for i in range(epochs):
    print(f"Epoch {i+1}\n--------------------------")
    train_loop(train_dataloader,model,loss_fn,optimizer)
    test_loop(test_dataloader,model,loss_fn)
print("Done")